# Manuscript Figure Generation

This notebook generates all main-text figures for the PEARL TB screening manuscript.

**Figures produced:**
- Figure 1: Calibration (Figure 1 style from manuscript)
- Figure 2: Projected trajectories (baseline vs scenario_3)
- Figure 3: Sensitivity analysis (hard-to-reach population)
- Figure 4: Screening algorithms and coverage comparison

All figures saved in high-resolution PNG and PDF format.

In [ ]:
# Import standard libraries
from pathlib import Path
from math import ceil
import numpy as np
import pandas as pd
import yaml

# Import scientific/plotting
import arviz as az
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.gridspec as gridspec
from PIL import Image
from pdf2image import convert_from_path

# Import project modules
from tbh.paths import REPO_ROOT_PATH
from tbh.model import get_tb_model
from tbh.plotting import plot_model_fit_with_uncertainty, plot_two_scenarios, plot_diff_outputs, title_lookup
import tbh.plotting as pl
import tbh.runner_tools as rt
from estival.model import BayesianCompartmentalModel

from importlib import reload

# Configure matplotlib
plt.style.use("seaborn-v0_8-white")
text_color = "#1f1f1f"
plt.rcParams.update({
    "font.family": "Helvetica",
    "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.5,
    "legend.frameon": False,
    "legend.fontsize": 7,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "text.color": text_color,
    "axes.labelcolor": text_color,
    "axes.titlecolor": text_color,
    "xtick.color": text_color,
    "ytick.color": text_color,
    "axes.edgecolor": text_color,
})

print("Imports successful")

## Configuration

In [ ]:
# Set paths
BASE_DIR = REPO_ROOT_PATH / "remote_cluster" / "outputs" / "59094989_new_priors"
OUTPUT_DIR = REPO_ROOT_PATH / "notebooks" / "manuscript_figs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Base-case configuration for main text
BASE_CASE_REL_SUS = 1.5
BASE_CASE_REG = 1.0

# Scenario names for labeling
SC_NAMES_CUSTOM = {
    'baseline': 'No screening',
    'scenario_1': 'PEARL / 65%',
    'scenario_2': 'PEARL / 75%',
    'scenario_3': 'PEARL / 85%',
    'scenario_6': 'CXR-TST / 65%',
    'scenario_7': 'CXR-TST / 75%',
    'scenario_8': 'CXR-TST / 85%',
    'scenario_16': 'CXR / 65%',
    'scenario_17': 'CXR / 75%',
    'scenario_18': 'CXR / 85%'
}

# Figure-specific outputs
TRAJECTORY_OUTPUTS = [
    "tb_incidence_per100k",
    "viable_tbi_prevalence_perc",
    "tb_prevalence_per100k",
    "tb_mortality_per100k",
]

print(f"Output directory: {OUTPUT_DIR}")
print(f"Base case: rel_sus={BASE_CASE_REL_SUS}, reg={BASE_CASE_REG}")

## Load task data

In [ ]:
# Scan directory for completed tasks
task_dirs = sorted([p for p in BASE_DIR.iterdir() if p.is_dir()])

records = []
for task_dir in task_dirs:
    files = [x for x in task_dir.iterdir() if x.is_file()]
    names = {f.name for f in files}
    
    has_idata = "idata.nc" in names
    has_details = "details.yaml" in names
    
    if len(files) == 0:
        status = "empty"
    elif has_details:
        status = "completed"
    elif has_idata:
        status = "partial"
    else:
        status = "failed"
    
    records.append({
        "task": task_dir.name,
        "task_path": task_dir,
        "n_files": len(files),
        "status": status,
    })

status_df = pd.DataFrame(records)
completed_df = status_df.loc[status_df["status"] == "completed"].copy()
print(f"Found {len(completed_df)} completed tasks")

In [ ]:
# Load configuration map for task parameters
config_map_path = BASE_DIR / "task_config_map.yaml"
with open(config_map_path, "r") as f:
    raw_task_config_map = yaml.safe_load(f)

config_source = raw_task_config_map.get("tasks", raw_task_config_map) if isinstance(raw_task_config_map, dict) else raw_task_config_map
task_to_config = config_source.get("task_to_config", {}) if isinstance(config_source, dict) else {}

def get_task_config(task_name):
    task_num = int(task_name.split("_")[1])
    return task_to_config.get(task_num, task_to_config.get(str(task_num)))

completed_df["task_config"] = completed_df["task"].apply(get_task_config)
print("Task configurations loaded")

In [ ]:
# Load base-case task bundle
def find_task_row(df, rel_sus=None, reg=None):
    x = df.copy()
    x["rel_sus_unreachable"] = pd.to_numeric(
        x["task_config"].apply(lambda d: d.get("rel_sus_unreachable") if isinstance(d, dict) else np.nan),
        errors="coerce",
    )
    x["clinical_regression_rate"] = pd.to_numeric(
        x["task_config"].apply(lambda d: d.get("clinical_regression_rate") if isinstance(d, dict) else np.nan),
        errors="coerce",
    )
    
    if rel_sus is not None:
        x = x[np.isclose(x["rel_sus_unreachable"], rel_sus)]
    if reg is not None:
        x = x[np.isclose(x["clinical_regression_rate"], reg)]
    
    if x.empty:
        raise ValueError(f"No task found for rel_sus={rel_sus}, reg={reg}")
    
    return x.sort_values("task").iloc[0]

base_task_row = find_task_row(completed_df, rel_sus=BASE_CASE_REL_SUS, reg=BASE_CASE_REG)
base_task_path = Path(base_task_row["task_path"])
base_task_name = base_task_row["task"]

print(f"Base case task: {base_task_name}")

# Load uncertainty dataframes for scenarios
unc_baseline = pd.read_parquet(base_task_path / "uncertainty_df_baseline.parquet")
unc_scenario3 = pd.read_parquet(base_task_path / "uncertainty_df_scenario_3.parquet")

print("Base-case uncertainty data loaded")

In [ ]:
# Load BCM for calibration figure
params, priors, tv_params = rt.get_parameters_and_priors()

with open(base_task_path / "details.yaml", "r") as f:
    docs = list(yaml.safe_load_all(f))

model_config = docs[1] if len(docs) > 1 else {}
if not isinstance(model_config, dict):
    model_config = {}

model = get_tb_model(model_config, tv_params)
bcm = BayesianCompartmentalModel(model, params, priors, rt.targets)

print("BCM loaded for calibration")

## Figure 1: Model Structure and Calibration

In [ ]:
reload(pl)

def make_figure_1_calibration(uncertainty_df, bcm, colour="#B22222", figsize=(10.8, 8.5), model_pdf_path=None):
    """Generate Figure 1: Model structure and calibration panel with uncertainty."""
    selected_outputs = [
        "pearl_posXreach_reachable_per100k",
        "cxr_posXreach_reachable_per100k",
        "perc_prev_subclinicalXreach_reachable",
        "perc_prev_infectiousXreach_reachable",
        "notifications",
    ]
    
    n_col = 3
    n_data_panels = len(selected_outputs) + 1  # 6 data panels
    n_data_row = ceil(n_data_panels / n_col)  # 2 rows for data
    
    # Create gridspec: 1 row for PDF + 2 rows for data panels
    fig = plt.figure(figsize=figsize)
    gs = gridspec.GridSpec(n_data_row + 1, n_col, figure=fig,
                           height_ratios=[1.8, 1, 1], hspace=.2, wspace=0.3)
    
    # Create PDF subplot (top row, spanning all columns)
    ax_pdf = fig.add_subplot(gs[0, :])
    ax_pdf.axis('off')  # Hide axes for image display
    
    # Load and display PDF if provided
    if model_pdf_path:
        try:
            images = convert_from_path(model_pdf_path, dpi=300)
            if images:
                ax_pdf.imshow(images[0])
        except Exception as e:
            ax_pdf.text(0.5, 0.5, f"Error loading PDF: {e}", ha='center', va='center')
    
    # Create axes for data panels
    axes = []
    for i in range(n_data_panels):
        row = 1 + (i // n_col)
        col = i % n_col
        ax = fig.add_subplot(gs[row, col])
        axes.append(ax)
    
    for i, output in enumerate(selected_outputs):
        ax = axes[i]
        x_min = 1995 if output == "notifications" else 2010
        pl.plot_model_fit_with_uncertainty(ax, uncertainty_df, output, bcm, x_lim=(x_min, 2025), colour=colour, target_ms=15)
        if i == 0:
            ax.legend()
    
    # Age-stratified TST positivity panel
    ax = axes[len(selected_outputs)]
    agegroups = ["3_9", "10", "15+", "18+"]
    model_median, model_low, model_high, observed, x_tick_labels = [], [], [], [], []
    
    for age in agegroups:
        output_name = f"tst_posXage_{age}Xreach_reachable_perc"
        year = bcm.targets[output_name].data.index[0]
        q = uncertainty_df[output_name].loc[year]
        obs = bcm.targets[output_name].data.iloc[0]
        
        model_median.append(q["0.5"])
        model_low.append(q["0.025"])
        model_high.append(q["0.975"])
        observed.append(obs)
        
        suffix = f" y.o.\n({year})"
        if age == "3_9":
            x_tick_labels.append("3-9" + suffix)
        elif age == "15+":
            x_tick_labels.append("15+" + suffix)
        else:
            x_tick_labels.append(f"{age}" + suffix)
    
    x = range(len(agegroups))
    ax.errorbar(
        [i - 0.06 for i in x],
        model_median,
        yerr=[
            [m - l for m, l in zip(model_median, model_low)],
            [h - m for h, m in zip(model_high, model_median)],
        ],
        fmt="D",
        color=colour,
        ecolor=colour,
        markersize=3,
        elinewidth=2.0,
        capsize=0,
        label="Model (median, 95% CI)",
    )
    ax.scatter([i + 0.06 for i in x], observed, color="black", s=7, zorder=5, label="Observed")
    ax.set_xticks(list(x))
    ax.set_xticklabels(x_tick_labels)
    ax.set_ylabel(title_lookup["tst_posXreach_reachable_perc"])
    
    model_handle = mlines.Line2D([], [], color=colour, marker="D", markersize=3, linestyle="-", label="Model (median, 95% CrI)")
    obs_handle = mlines.Line2D([], [], color="black", marker="o", linestyle="None", markersize=3, label="Observed")
    ax.legend(handles=[obs_handle, model_handle], frameon=False, loc="best")
    
    # Panel letters - 'a)' for PDF, 'b)'-'g)' for data panels
    # Add letter to PDF
    letter_fontsize = 9
    ax_pdf.text(-0.05, 0.98, "a)", transform=ax_pdf.transAxes, fontsize=letter_fontsize, fontweight="bold", va="top")
    
    # Add letters to data panels
    for i, ax in enumerate(axes):
        letter_idx = i + 1  # Start from 'b)'
        ax.text(
            -0.15,
            1.1,
            f"{chr(97 + letter_idx)})",
            transform=ax.transAxes,
            fontsize=letter_fontsize,
            fontweight="bold",
            va="top",
        )
    
    return fig

# Generate Figure 1
model_pdf = REPO_ROOT_PATH / "notebooks" / "manuscript_figs" / "tb_model.pdf"
fig1 = make_figure_1_calibration(unc_baseline, bcm, model_pdf_path=model_pdf)
plt.savefig(OUTPUT_DIR / "figure_1_calibration.png", dpi=300, bbox_inches='tight')
plt.savefig(OUTPUT_DIR / "figure_1_calibration.pdf", bbox_inches='tight')
plt.show()

print("Figure 1 saved")

## Figure 2: Projected trajectories

In [ ]:
reload(pl)
# Load scenario_3 uncertainty data
unc_dfs = {
    "baseline": unc_baseline,
    "scenario_3": unc_scenario3,
}

# Generate Figure 2
unc_sc_colours = ["#B22222", "#54992c"]

fig, axes = plt.subplots(2, 2, figsize=(7, 4.65), sharex=False)
axes = axes.flatten()

for ax, output in zip(axes, TRAJECTORY_OUTPUTS):
    pl.plot_two_scenarios(
        ax,
        unc_dfs,
        output,
        scenarios=["baseline", "scenario_3"],
        xlim=(2020, 2035),
        include_unc=True,
        ylab_fontsize=9,
        unc_sc_colours=unc_sc_colours,
        include_legend=ax==axes[0],
        sc_names=SC_NAMES_CUSTOM
    )
    # ax.set_title(title_lookup.get(output, output), fontsize=10)

# Panel letters
panel_letters = [f"{chr(97 + i)})" for i in range(len(TRAJECTORY_OUTPUTS))]
for i, letter in enumerate(panel_letters):
    axes[i].text(
        -0.15,
        1.05,
        letter,
        transform=axes[i].transAxes,
        fontsize=9,
        fontweight="bold",
        va="top",
    )

fig.tight_layout()
plt.savefig(OUTPUT_DIR / "figure_2_trajectories.png", dpi=300, bbox_inches='tight')
plt.savefig(OUTPUT_DIR / "figure_2_trajectories.pdf", bbox_inches='tight')
plt.show()

print("Figure 2 saved")

## Figure 3: Sensitivity to hard-to-reach population

In [ ]:
# Part A: Proportion of TB incidence from hard-to-reach (2026 vs 2028)
scenario = "scenario_3"
years_to_plot = [2026, 2028]
target_rel_sus_values = [1.0, 1.5, 2.0, 3.0]
target_reg = 1.0
output_name = "prop_tb_incidenceXreach_unreachable"

rows = []
for _, row in completed_df.iterrows():
    task_path = Path(row["task_path"])
    task_cfg = row.get("task_config", {})
    
    if not isinstance(task_cfg, dict):
        continue
    
    rel_sus = pd.to_numeric(task_cfg.get("rel_sus_unreachable"), errors="coerce")
    reg = pd.to_numeric(task_cfg.get("clinical_regression_rate"), errors="coerce")
    
    if pd.isna(rel_sus) or pd.isna(reg):
        continue
    if not np.isclose(reg, target_reg):
        continue
    if rel_sus not in target_rel_sus_values:
        continue
    
    unc_file = task_path / f"uncertainty_df_{scenario}.parquet"
    if not unc_file.exists():
        continue
    
    unc_df = pd.read_parquet(unc_file)
    if output_name not in unc_df.columns:
        continue
    
    for yr in years_to_plot:
        if yr not in unc_df.index:
            continue
        
        q = unc_df[output_name].loc[yr]
        median = q["0.5"] if "0.5" in q.index else q[0.5]
        low = q["0.025"] if "0.025" in q.index else q[0.025]
        high = q["0.975"] if "0.975" in q.index else q[0.975]
        
        rows.append({
            "rel_sus_unreachable": float(rel_sus),
            "year": int(yr),
            "median_pct": 100.0 * float(median),
            "low_pct": 100.0 * float(low),
            "high_pct": 100.0 * float(high),
        })

plot_df_fig3a = pd.DataFrame(rows)
plot_df_fig3a = plot_df_fig3a.groupby(["rel_sus_unreachable", "year"], as_index=False)[["median_pct", "low_pct", "high_pct"]].median()
plot_df_fig3a = plot_df_fig3a.sort_values(["rel_sus_unreachable", "year"]).reset_index(drop=True)

print("Figure 3a data prepared")

In [ ]:
# Part B: Impact sensitivity across rel_sus and clinical regression rates
sc_id_list = [3]  # scenario_3

for sc_id in sc_id_list:
    scenario_data = []
    
    for _, row in completed_df.iterrows():
        task_name = row["task"]
        task_path = Path(row["task_path"])
        task_cfg = row.get("task_config", {})
        
        diff_file = task_path / f"diff_quantiles_df_ref_baseline_scenario_{sc_id}.parquet"
        if diff_file.exists():
            df_diff = pd.read_parquet(diff_file)
            scenario_data.append({
                "task": task_name,
                "rel_sus_unreachable": task_cfg.get("rel_sus_unreachable") if isinstance(task_cfg, dict) else None,
                "clinical_regression_rate": task_cfg.get("clinical_regression_rate") if isinstance(task_cfg, dict) else None,
                "TB_averted_relative": df_diff.loc[0.5, "TB_averted_relative"],
                "TB_averted_relative_low": df_diff.loc[0.025, "TB_averted_relative"],
                "TB_averted_relative_high": df_diff.loc[0.975, "TB_averted_relative"],
            })
    
    scenario_df = pd.DataFrame(scenario_data)
    
    for col in [
        "rel_sus_unreachable",
        "clinical_regression_rate",
        "TB_averted_relative",
        "TB_averted_relative_low",
        "TB_averted_relative_high",
    ]:
        scenario_df[col] = pd.to_numeric(scenario_df[col], errors="coerce")
    
    scenario_df = scenario_df.dropna(
        subset=[
            "rel_sus_unreachable",
            "clinical_regression_rate",
            "TB_averted_relative",
            "TB_averted_relative_low",
            "TB_averted_relative_high",
        ]
    ).copy()
    
    scenario_df = scenario_df.sort_values(
        ["clinical_regression_rate", "rel_sus_unreachable"]
    ).reset_index(drop=True)
    scenario_df["x"] = range(len(scenario_df))
    
    plot_df_fig3b = scenario_df

print("Figure 3b data prepared")

In [ ]:
# Generate Figure 3: Combined sensitivity analysis (bar plot and line plot)
year_colors = {2026: "#EF8775", 2028: "#b43232"}
rel_sus_palette = {
    1.0: "#127624",
    1.5: "#094fa5",
    2.0: "#6d32a4",
    3.0: "#a81334",
}
default_rel_sus_color = "#7f7f7f"

fig, axes = plt.subplots(2, 1, figsize=(7, 7))

# Panel A: Bar plot (2026 vs 2028)
ax = axes[0]
x = np.arange(len(target_rel_sus_values), dtype=float)
bar_width = 0.35

year_offsets = {
    years_to_plot[0]: -bar_width / 2.0,
    years_to_plot[1]: bar_width / 2.0,
}

leg_label = {2026: "Before screening (2026)", 2028: "After screening (2028)"}

for yr in years_to_plot:
    sub = (
        plot_df_fig3a.loc[plot_df_fig3a["year"] == yr]
        .set_index("rel_sus_unreachable")
        .reindex(target_rel_sus_values)
        .reset_index()
    )
    
    y = sub["median_pct"].to_numpy()
    y_low = sub["low_pct"].to_numpy()
    y_high = sub["high_pct"].to_numpy()
    
    x_pos = x + year_offsets[yr]
    
    ax.bar(
        x_pos,
        y,
        width=bar_width,
        color=year_colors.get(yr, "#333333"),
        alpha=0.9,
        label=leg_label[yr],
        zorder=2,
    )
    ax.errorbar(
        x_pos,
        y,
        yerr=[y - y_low, y_high - y],
        fmt="none",
        ecolor="black",
        elinewidth=1.2,
        capsize=3,
        zorder=3,
    )

ax.set_xticks(x)
ax.set_xticklabels([f"{v:.1f}" for v in target_rel_sus_values], fontsize=7)
ax.set_xlabel("Relative susceptibility to M.tb infection among the hard-to-reach", fontsize=9)
ax.set_ylabel("% of total TB incidence from the 15% hard-to-reach", fontsize=9)
ax.grid(axis="y", alpha=0.35, zorder=1)
ax.legend(frameon=False, fontsize=9)
ax.text(-0.06, 1.08, "a)", transform=ax.transAxes, fontsize=9, fontweight='bold', va='top')

# Panel B: Line plot
ax = axes[1]
rel_sus_values = sorted(plot_df_fig3b["rel_sus_unreachable"].unique())
for rel_sus in rel_sus_values:
    sub = plot_df_fig3b.loc[
        plot_df_fig3b["rel_sus_unreachable"] == rel_sus
    ].sort_values("x")
    
    xs = sub["x"].to_numpy()
    y = 100.0 * sub["TB_averted_relative"].to_numpy()
    y_low = 100.0 * sub["TB_averted_relative_low"].to_numpy()
    y_high = 100.0 * sub["TB_averted_relative_high"].to_numpy()
    
    series_color = rel_sus_palette.get(float(rel_sus), default_rel_sus_color)
    
    ax.errorbar(
        xs,
        y,
        yerr=[y - y_low, y_high - y],
        fmt="o",
        color=series_color,
        ecolor=series_color,
        markersize=5,
        elinewidth=1.5,
        capsize=4,
        zorder=3,
        label=f"rel_sus = {rel_sus:.1f}",
    )
    if len(xs) > 1:
        ax.plot(xs, y, color=series_color, linewidth=0.5, alpha=0.9, zorder=2)

# Vertical separators after datapoints 4, 8, and 12
for sep in [3.5, 7.5, 11.5]:
    if sep < len(plot_df_fig3b) - 0.5:
        ax.axvline(sep, color="0.5", linestyle="--", linewidth=1.0, alpha=0.8, zorder=1)

x_all = plot_df_fig3b["x"].to_numpy()
ax.set_xticks(x_all)
ax.set_xticklabels([f"{v:.1f}" for v in plot_df_fig3b["rel_sus_unreachable"]], rotation=0, fontsize=7)
ax.set_xlabel("Relative susceptibility to M.tb infection among the hard-to-reach", fontsize=9)
ax.set_ylabel("% TB episodes averted by PEARL over 2026-35", fontsize=9)
ax.grid(axis="y", linestyle="-", linewidth=0.7, alpha=0.4)
# ax.legend(frameon=False, fontsize=7, loc='upper left')
ax.text(-0.06, 1.05, "b)", transform=ax.transAxes, fontsize=9, fontweight='bold', va='top')

# Group labels for clinical regression rate
group_size = 4
reg_values = plot_df_fig3b["clinical_regression_rate"].drop_duplicates().tolist()
for i, reg in enumerate(reg_values):
    start = i * group_size
    end = min(start + group_size - 1, len(plot_df_fig3b) - 1)
    if start <= end:
        center = (start + end) / 2.0
        ax.text(
            center,
            0.99,
            f"TB state regression rate: {reg:.1f}/y",
            ha="center",
            va="bottom",
            transform=ax.get_xaxis_transform(),
            fontsize=7,
            color=text_color,
        )

fig.tight_layout()
plt.savefig(OUTPUT_DIR / "figure_3_sensitivity_analysis.png", dpi=300, bbox_inches='tight')
plt.savefig(OUTPUT_DIR / "figure_3_sensitivity_analysis.pdf", bbox_inches='tight')
plt.show()

print("Figure 3 (combined panels a and b) saved")

## Figure 4: Algorithm and coverage comparison

In [ ]:
# Load diff outputs for algorithm comparison
scenarios_to_compare = ["scenario_1", "scenario_2","scenario_3", "scenario_6", "scenario_7", "scenario_8", "scenario_16", "scenario_17", "scenario_18"]
diff_dfs = {}

for scenario in scenarios_to_compare:
    diff_path = base_task_path / f"diff_quantiles_df_ref_baseline_{scenario}.parquet"
    if diff_path.exists():
        diff_dfs[scenario] = pd.read_parquet(diff_path)

print(f"Loaded {len(diff_dfs)} scenario diff files for Figure 4")

In [ ]:
# Generate Figure 4: Algorithm and coverage comparison
reload(pl)

fig, axes = plt.subplots(2, 1, figsize=(5, 5), sharex=False)

group_labels = ["PEARL", "Without Xpert", "Without Xpert & TST"]
coverage_labels = ["65%", "75%", "85%"]
xtick_labels = coverage_labels * len(group_labels)

def format_fig4_axis(ax):
    # Keep only coverage on x-axis and draw separators between algorithm groups.
    ax.set_xticks(range(1, len(scenarios_to_compare) + 1), xtick_labels)
    plt.setp(ax.get_xticklabels(), rotation=0, ha="center")

    for boundary in (3.5, 6.5):
        ax.axvline(boundary, color="0.5", linestyle="--", linewidth=0.9, alpha=0.9, zorder=0)

    group_centers = [2, 5, 8]
    for center, label in zip(group_centers, group_labels):
        ax.text(
            center,
            .98,
            label,
            ha="center",
            va="bottom",
            transform=ax.get_xaxis_transform(),
            fontsize=7,
        )
    ax.set_xlabel("Screening coverage")
    ax.set_ylim(0, 65)

# Panel A: TB episodes averted
ax = axes[0]
pl.plot_diff_outputs(ax, diff_dfs, "TB_averted_relative", scenarios_to_compare, colour="#B22222")
format_fig4_axis(ax)
ax.grid(axis="y", linestyle="-", linewidth=0.7, alpha=0.4)
ax.set_title("a)", loc='left')

# Panel B: TB deaths averted
ax = axes[1]
pl.plot_diff_outputs(ax, diff_dfs, "deaths_averted_relative", scenarios_to_compare, colour="#B22222")
format_fig4_axis(ax)
ax.grid(axis="y", linestyle="-", linewidth=0.7, alpha=0.4)
ax.set_title("b)", loc='left')

fig.tight_layout()
plt.savefig(OUTPUT_DIR / "figure_4_algorithms_coverage.png", dpi=300, bbox_inches='tight')
plt.savefig(OUTPUT_DIR / "figure_4_algorithms_coverage.pdf", bbox_inches='tight')
plt.show()

print("Figure 4 saved")

## Results text auto-population

In [ ]:
# Auto-populate the full Results-section text using estimates from the selected output folder.
# Assumes earlier cells have defined: base_dir, completed_df.

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def _load_unc(task_path, scenario):
    p = task_path / f"uncertainty_df_{scenario}.parquet"
    if not p.exists():
        raise FileNotFoundError(f"Missing uncertainty file: {p}")
    return pd.read_parquet(p)


def _load_diff(task_path, scenario):
    p = task_path / f"diff_quantiles_df_ref_baseline_{scenario}.parquet"
    if not p.exists():
        raise FileNotFoundError(f"Missing diff file: {p}")
    return pd.read_parquet(p)


def _q(unc_df, output, year, quantile):
    q_key = str(quantile)
    return float(unc_df.loc[float(year), (output, q_key)])


def _cri(unc_df, output, year, scale=1.0, dec=0):
    med = scale * _q(unc_df, output, year, 0.5)
    low = scale * _q(unc_df, output, year, 0.025)
    high = scale * _q(unc_df, output, year, 0.975)
    f = f"{{:.{dec}f}}"
    return f.format(med), f.format(low), f.format(high)


def _find_task_row(df, rel_sus=None, reg=None):
    x = df.copy()
    x["rel_sus_unreachable"] = pd.to_numeric(
        x["task_config"].apply(lambda d: d.get("rel_sus_unreachable") if isinstance(d, dict) else np.nan),
        errors="coerce",
    )
    x["clinical_regression_rate"] = pd.to_numeric(
        x["task_config"].apply(lambda d: d.get("clinical_regression_rate") if isinstance(d, dict) else np.nan),
        errors="coerce",
    )

    if rel_sus is not None:
        x = x[np.isclose(x["rel_sus_unreachable"], rel_sus)]
    if reg is not None:
        x = x[np.isclose(x["clinical_regression_rate"], reg)]

    if x.empty:
        raise ValueError(f"No task found for rel_sus={rel_sus}, reg={reg}")

    return x.sort_values("task").iloc[0]


def _pct_averted(diff_df, col):
    return (
        100.0 * float(diff_df.loc[0.5, col]),
        100.0 * float(diff_df.loc[0.025, col]),
        100.0 * float(diff_df.loc[0.975, col]),
    )


# Choose base-case task for manuscript text.
# Current convention here: rel_sus_unreachable=1.5 and clinical_regression_rate=1.0.
base_task = _find_task_row(completed_df, rel_sus=1.5, reg=1.0)
base_task_name = base_task["task"]
base_task_path = Path(base_task["task_path"])

unc_baseline = _load_unc(base_task_path, "baseline")
unc_sc3 = _load_unc(base_task_path, "scenario_3")
diff_sc3 = _load_diff(base_task_path, "scenario_3")

# Core burden estimates at the beginning of 2026 (before screening)
prev_2026_med, prev_2026_low, prev_2026_high = _cri(unc_baseline, "tb_prevalence_per100k", 2026, scale=1.0, dec=0)
inc_2026_med, inc_2026_low, inc_2026_high = _cri(unc_baseline, "tb_incidence_per100k", 2026, scale=1.0, dec=0)
vtbi_2026_med, vtbi_2026_low, vtbi_2026_high = _cri(unc_baseline, "viable_tbi_prevalence_perc", 2026, scale=1.0, dec=0)

# 2028 vs 2026 reductions under PEARL-like screening (scenario_3)
prev_2026_sc3 = _q(unc_sc3, "tb_prevalence_per100k", 2026, 0.5)
prev_2028_sc3 = _q(unc_sc3, "tb_prevalence_per100k", 2028, 0.5)
vtbi_2026_sc3 = _q(unc_sc3, "viable_tbi_prevalence_perc", 2026, 0.5)
vtbi_2028_sc3 = _q(unc_sc3, "viable_tbi_prevalence_perc", 2028, 0.5)

prev_drop_pct = 100.0 * (prev_2026_sc3 - prev_2028_sc3) / prev_2026_sc3
vtbi_drop_pct = 100.0 * (vtbi_2026_sc3 - vtbi_2028_sc3) / vtbi_2026_sc3

# 2028 incidence and mortality under screening
inc_2028_sc3_med, _, _ = _cri(unc_sc3, "tb_incidence_per100k", 2028, scale=1.0, dec=0)
mort_2028_sc3_med, _, _ = _cri(unc_sc3, "tb_mortality_per100k", 2028, scale=1.0, dec=0)

# 2035 incidence with and without screening
inc_2035_sc3_med, _, _ = _cri(unc_sc3, "tb_incidence_per100k", 2035, scale=1.0, dec=0)
inc_2035_base_med, _, _ = _cri(unc_baseline, "tb_incidence_per100k", 2035, scale=1.0, dec=0)

# Cumulative impact during 2026-35 (from diff outputs)
tb_av_med, tb_av_low, tb_av_high = _pct_averted(diff_sc3, "TB_averted_relative")
d_av_med, d_av_low, d_av_high = _pct_averted(diff_sc3, "deaths_averted_relative")

# Sensitivity: impact at rel_sus 1.0 vs 3.0, fixing clinical_regression_rate=1.0
task_rel1 = _find_task_row(completed_df, rel_sus=1.0, reg=1.0)
task_rel3 = _find_task_row(completed_df, rel_sus=3.0, reg=1.0)

diff_rel1 = _load_diff(Path(task_rel1["task_path"]), "scenario_3")
diff_rel3 = _load_diff(Path(task_rel3["task_path"]), "scenario_3")
unc_rel3_sc3 = _load_unc(Path(task_rel3["task_path"]), "scenario_3")

rel1_med, rel1_low, rel1_high = _pct_averted(diff_rel1, "TB_averted_relative")
rel3_med, rel3_low, rel3_high = _pct_averted(diff_rel3, "TB_averted_relative")

# Contribution of hard-to-reach population to TB incidence (figure 3 text)
unreach_before_base = 100.0 * _q(unc_baseline, "prop_tb_incidenceXreach_unreachable", 2026, 0.5)
unreach_after_base = 100.0 * _q(unc_sc3, "prop_tb_incidenceXreach_unreachable", 2028, 0.5)
unreach_before_rel3 = 100.0 * _q(unc_rel3_sc3, "prop_tb_incidenceXreach_unreachable", 2026, 0.5)
unreach_after_rel3 = 100.0 * _q(unc_rel3_sc3, "prop_tb_incidenceXreach_unreachable", 2028, 0.5)

# Algorithm and coverage comparisons (same base-case task)
diff_sc1 = _load_diff(base_task_path, "scenario_1")   # PEARL | 65%
diff_sc3 = _load_diff(base_task_path, "scenario_3")   # PEARL | 85% (84.5%)
diff_sc6 = _load_diff(base_task_path, "scenario_6")   # CXR-TST | 65%
diff_sc8 = _load_diff(base_task_path, "scenario_8")   # CXR-TST | 85% (84.5%)
diff_sc18 = _load_diff(base_task_path, "scenario_18") # Disease screening only | 85% (84.5%)

pearl85_med, _, _ = _pct_averted(diff_sc3, "TB_averted_relative")
cxrtst85_med, _, _ = _pct_averted(diff_sc8, "TB_averted_relative")
abs_diff_85 = abs(pearl85_med - cxrtst85_med)

disease85_med, _, _ = _pct_averted(diff_sc18, "TB_averted_relative")
pearl65_med, _, _ = _pct_averted(diff_sc1, "TB_averted_relative")
cxrtst65_med, _, _ = _pct_averted(diff_sc6, "TB_averted_relative")

results_text = f"""
Model calibration and estimated TB burden

The model adequately reproduced the epidemiological patterns used for calibration across model configurations that varied the susceptibility of the hard-to-reach population and rates of regression across the TB disease spectrum (figure 1; appendix pp XX-XX). Posterior estimates were consistent with observed TB prevalence, disease-state distributions, historical notifications, and age-specific TST positivity.

Calibration substantially reduced uncertainty in parameters governing transmission, progression across the TB infection-disease spectrum, loss of immune containment, and passive detection. The infection-clearance rate was also constrained by the data, although less strongly than these other parameters (appendix pp XX-XX).

At the beginning of 2026, before screening, the estimated prevalence of TB across the four disease states was {prev_2026_med} per 100 000 population (95% CrI {prev_2026_low}-{prev_2026_high}), and estimated TB incidence was {inc_2026_med} per 100 000 person-years ({inc_2026_low}-{inc_2026_high}). An estimated {vtbi_2026_med}% ({vtbi_2026_low}-{vtbi_2026_high}) of the population had viable M tuberculosis infection, defined as infection with the potential to progress to TB disease. These estimates represented the entire modelled population, including screening-reachable and hard-to-reach individuals of all ages, and were therefore not directly comparable with prevalence estimates from the completed phase of PEARL screening.

Projected impact of PEARL-like screening

In the base-case model, implementation of the PEARL algorithm at 85% population coverage produced immediate reductions in TB prevalence and viable infection prevalence, which were approximately {prev_drop_pct:.0f}% and {vtbi_drop_pct:.0f}% lower, respectively, in 2028 than in 2026 (figure 2). TB incidence and mortality also declined, reaching {inc_2028_sc3_med} per 100 000 person-years and {mort_2028_sc3_med} per 100 000 person-years, respectively, in 2028.

These reductions were projected to be sustained. In 2035, TB incidence was projected to be {inc_2035_sc3_med} per 100 000 person-years with PEARL-like screening, compared with {inc_2035_base_med} per 100 000 person-years without active screening. Over 2026-35, PEARL-like screening was projected to avert {tb_av_med:.0f}% ({tb_av_low:.0f}-{tb_av_high:.0f}) of TB episodes and {d_av_med:.0f}% ({d_av_low:.0f}-{d_av_high:.0f}) of TB deaths expected without active screening.

Sensitivity to the population missed by screening and disease-spectrum assumptions

Projected screening impact was most sensitive to assumptions about the susceptibility of individuals hard-to-reach by screening (figure 3). At 85% coverage, PEARL-like screening averted {rel1_med:.0f}% ({rel1_low:.0f}-{rel1_high:.0f}) of TB episodes during 2026-35 when hard-to-reach and screening-reachable individuals were assumed to have equal susceptibility to M tuberculosis infection, compared with {rel3_med:.0f}% ({rel3_low:.0f}-{rel3_high:.0f}) when hard-to-reach individuals were assumed to have three times the susceptibility of screening-reachable individuals.

Although only 15% of the population was assumed to be hard-to-reach, this group accounted for {unreach_before_base:.0f}% of total TB incidence before PEARL-like screening and {unreach_after_base:.0f}% afterwards in the base case, assuming this population was 50% more susceptible than the screening-reachable population. When assuming three-times greater susceptibility to infection, its contribution rose from {unreach_before_rel3:.0f}% before screening to {unreach_after_rel3:.0f}% after screening (figure 3).

By contrast, projected screening impact was only weakly sensitive to assumptions about rates of movement across the TB infection-disease spectrum (figure 3; appendix pp XX-XX).

Comparison of screening algorithms and coverage

Among strategies that retained TST and preventive treatment, population coverage had a greater influence on projected impact than the inclusion of Xpert in frontline screening (figure 4; table 1). At equivalent coverage, the PEARL and CXR-TST algorithms produced similar reductions in cumulative TB episodes and deaths. At 85% coverage, the proportions of TB episodes averted during 2026-35 differed by only {abs_diff_85:.1f} percentage points between these algorithms, indicating that removal of Xpert from frontline screening had only a modest effect on projected population-level impact.

In contrast, removing TST and preventive treatment substantially reduced both the magnitude and durability of the intervention effect. Even at 85% coverage, disease screening alone averted only {disease85_med:.0f}% of TB episodes during 2026-35, less than the {pearl65_med:.0f}% and {cxrtst65_med:.0f}% averted by the PEARL and CXR-TST algorithms, respectively, at 65% coverage. Similar patterns were observed for cumulative TB deaths (figure 4). These findings indicate that maintaining the identification and preventive treatment of viable infection was more important to sustained population-level impact than retaining Xpert as a frontline screening test.

Additional sensitivity analyses

Projected intervention effects were qualitatively unchanged when TPT completion was reduced to 60%, when 50% of prevalent TB was assumed to be subclinical, and when homogeneous age mixing was assumed (appendix pp XX-XX). However, the homogeneous age-mixing model resulted in poorer agreement with the calibration data, particularly age-specific TST positivity, indicating that age-assortative mixing was important for reproducing observed age-specific patterns of M tuberculosis transmission.
""".strip()

display(Markdown("### Auto-populated Results text"))
display(Markdown(f"Base-case task used: **{base_task_name}**"))
display(Markdown(results_text))

# Save to text file
with open(OUTPUT_DIR / "results_text.txt", "w") as f:
    f.write(results_text)

print("\nResults text saved to results_text.txt")